# Defining groups of genetic clades with and without each subclonal copy number change of interest

## Import R libraries

In [2]:
library("data.table")
library("readr")
library("dplyr")
library("stringr")

## Options

In [8]:
options(scipen=999)

## Define groups based on a threshold of 80% of the clade with (without) the CNA of interest

In [20]:
# Import clade information

clades <- read_csv("../Data/CNA_Data/Processed/Genetic_Clades_And_SuperClades.csv",
                  col_types = "cdd")
rownames(clades) <- clades$Cell_id

# Import CNA data

df <- read_tsv("../Data/CNA_Data/Processed/CNA_Profiles_Segmented.tsv",
               col_types = "ccddd")
df <- df %>% dplyr::mutate(Location = sprintf("%s:%s-%s",chrom,start,end))

# Alterations of interest (breakpoints defined here according to the segmentation, i.e., more accurate breakpoints obtained later)

CNAs <- data.frame("Location" = c("chr1:162000001-175000000",
                                  "chr5:49500001-92500000",
                                  "chr18:1000001-13500000",
                                  "chr20:500001-19500000",
                                  "chrX:500001-53500000"),
                   "Alt_State" = c(1,3,4,4,1),
                   "OG_State" = c(2,2,2,3,2))

# Define CNA groups
groups <-c()
for (alt in CNAs$Location){
  for (clade in 1:length(unique(clades$Clade))){
    temp <- df %>% filter(cell_id %in% clades[clades$Clade == clade,]$Cell_id & Location == alt)
    temp2 <- temp %>% filter(state == CNAs[CNAs$Location == alt,]$Alt_State)
    temp3 <- temp %>% filter(state == CNAs[CNAs$Location == alt,]$OG_State)
    
    if (nrow(temp2) >= 0.8*nrow(temp)){
      groups <- rbind(groups,cbind("Location" = alt, "Clade" = clade, "Altered" = 1, "Opposing" = nrow(temp) - nrow(temp2)))
      
    } else if (nrow(temp3) >= 0.8*nrow(temp)){
      groups <- rbind(groups,cbind("Location" = alt, "Clade" = clade, "Altered" = 0, "Opposing" = nrow(temp) - nrow(temp3)))
    } else {
      groups <- rbind(groups,cbind("Location" = alt, "Clade" = clade, "Altered" = NA, "Opposing" = nrow(temp)))
    }
  }
}

counts <- data.frame(table(clades$Clade))
colnames(counts) <- c("Clade","Count")

groups_new <- data.frame(groups) %>% left_join(counts, by = "Clade")

groups_final <- na.omit(groups_new)

write_csv(groups_final, "../Data/CNA_Data/Processed/GeneticCladeGroupings_DifferentialATACSignalAnalysis.csv")

Warning message:
“Setting row names on a tibble is deprecated.”


We can recover the total number of cells in each group (Total_cells_0, Total_cell_1) as well as the total number (Total_opposing_0, Total_opposing_1)/proportion (Prop_Opposing_0, Prop_Opposing_1) of cells whose karyotype does not conform to their assigned group:

In [18]:
result <- groups_final %>%
  group_by(Location, Altered) %>%
  summarise(
    Total_cells     = sum(Count, na.rm = TRUE),
    Total_opposing  = sum(as.numeric(Opposing), na.rm = TRUE),
    .groups = "drop"
  ) %>%
  tidyr::pivot_wider(
    names_from = Altered,
    values_from = c(Total_cells, Total_opposing),
    names_sep = "_"
  ) %>% mutate(Prop_Opposing_0 = Total_opposing_0/Total_cells_0) %>% 
  mutate(Prop_Opposing_1 = Total_opposing_1/Total_cells_1)

result

Location,Total_cells_0,Total_cells_1,Total_opposing_0,Total_opposing_1,Prop_Opposing_0,Prop_Opposing_1
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
chr18:1000001-13500000,292,151,16,15,0.054794521,0.099337748
chr1:162000001-175000000,387,83,9,5,0.023255814,0.060240964
chr20:500001-19500000,218,293,1,16,0.004587156,0.054607509
chr5:49500001-92500000,211,314,2,2,0.009478673,0.006369427
chrX:500001-53500000,342,182,2,0,0.005847953,0.000000000
